### 1. 셀레니움
셀레니움(Selenium)은 파이썬 코드로 실제 웹 브라우저(Chrome, Edge 등)를 자동으로 제어할 수 있게 해주는 웹 자동화 라이브러리입니다. 일반적인 requests 크롤링은 서버에서 받은 HTML만 분석하지만, Selenium은 브라우저를 직접 실행하여 버튼 클릭, 검색 입력, 스크롤, 로그인, 페이지 이동 등의 사용자 동작을 그대로 수행할 수 있기 때문에 JavaScript로 동적으로 생성되는 데이터까지 가져올 수 있습니다. 따라서 멜론, 유튜브, 쇼핑몰처럼 JavaScript 렌더링이 많은 사이트의 크롤링이나 웹 자동화 테스트에서 매우 많이 사용됩니다. 보통 webdriver.Chrome()으로 브라우저를 실행하고, find_element()로 요소를 찾으며, click(), send_keys() 등을 통해 자동화 작업을 수행합니다.



In [ ]:
import requests
from bs4 import BeautifulSoup

url = "http://127.0.0.1:5500/7.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
fruits = soup.select(".fruit")
print(fruits)

[]


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

In [5]:
driver = webdriver.Chrome()
url = "http://127.0.0.1:5500/7.html"
driver.get(url)

# JavaScript 실행 대기
time.sleep(2)
fruits = driver.find_elements(By.CLASS_NAME, "fruit")

for fruit in fruits:
    print(fruit.text)

driver.quit()

사과
바나나
오렌지
딸기


> 👉 find_elements()는 Selenium에서 HTML 요소를 여러 개 찾을 때 사용하는 메서드입니다. 하나의 요소만 찾는 find_element()와 달리, 조건에 맞는 요소들을 리스트 형태로 모두 반환합니다. 예를 들어 여러 개의 이미지, 게시글, 테이블 행(tr) 등을 반복해서 크롤링할 때 매우 자주 사용됩니다. 찾는 방식은 By.ID, By.CLASS_NAME, By.CSS_SELECTOR, By.XPATH 등 다양한 선택자를 사용할 수 있으며, 반환 결과는 리스트이므로 for문으로 반복 처리하는 경우가 많습니다. 또한 find_element()는 요소를 찾지 못하면 에러가 발생하지만, find_elements()는 빈 리스트([])를 반환하기 때문에 반복 크롤링에서 더 안정적으로 사용되는 경우도 많습니다.

### 2. 멜론 아티스트 곡 리스트 크롤링

In [2]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

### ※ xpath
XPath는 XML 또는 HTML 문서 내에서 특정 요소나 속성을 선택하기 위해 사용되는 경로 표현 언어입니다. 웹 크롤링이나 자동화 도구에서 주로 사용되며, 요소를 효율적으로 찾을 수 있도록 도와줍니다. 일반적인 XPath는 특정 위치나 속성을 기준으로 요소를 선택하는 상대적인 경로를 사용합니다. 또한 full xpath는 루트 요소에서 시작하여 대상 요소까지의 절대적인 경로를 나타냅니다. 따라서 문서 구조가 변경되면 경로가 깨질 가능성이 높습니다.

In [1]:
def melon_search_from_main(keyword):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )
        song_tab.click()

        time.sleep(3)

        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )

        rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

        print("찾은 행 개수:", len(rows))

        for row in rows:
            try:
                cols = row.find_element(By.TAG_NAME, "td")
                if len(cols) < 5:
                    continue

                title_text = cols[2].text.strip()
                title_lines = [
                    line.strip()
                    for line in title_text.split("\n")
                    if line.strip()
                ]

                title = ""

                for line in title_lines:
                    if (
                        "재생" not in line
                        and "담기" not in line
                        and "상세정보" not in line
                        and not line.startswith("Title")
                    ):
                        title = line
                        break

                # Title 줄이 더 정확한 경우 보정
                for line in title_lines:
                    if line.startswith("Title "):
                        title = line.replace("Title ", "").strip()
                        break

                # 아티스트
                artist_text = cols[3].text.strip()
                artist_lines = [
                    line.strip()
                    for line in artist_text.split("\n")
                    if line.strip()
                ]
                artist = artist_lines[0] if artist_lines else ""

                # 앨범
                album_text = cols[4].text.strip()
                album_lines = [
                    line.strip()
                    for line in album_text.split("\n")
                    if line.strip()
                ]
                album = album_lines[0] if album_lines else ""

                # 좋아요 수
                like = ""
                try:
                    like = row.find_element(
                        By.CSS_SELECTOR,
                        "button.like span.cnt"
                        # #frm_defaultList > div > table > tbody > tr:nth-child(1) > td:nth-child(6) > div > button > span.cnt
                    ).text.strip()
                except:
                    pass

                if title:
                    data.append({
                        "곡명": title,
                        "아티스트": artist,
                        "앨범": album,
                        "좋아요수": like
                    })

            except Exception as e:
                print("행 처리 오류:", e)
                
        df = pd.DataFrame(data)
        if not df.empty:
            df.index = df.index + 1 
        
        file_name = f"melon_{keyword}_songs.csv"
        df.to_csv(file_name, encoding="utf-8-sig")

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

In [5]:
melon_search_from_main("신의 키스")

찾은 행 개수: 6
행 처리 오류: object of type 'WebElement' has no len()
행 처리 오류: object of type 'WebElement' has no len()
행 처리 오류: object of type 'WebElement' has no len()
행 처리 오류: object of type 'WebElement' has no len()
행 처리 오류: object of type 'WebElement' has no len()
행 처리 오류: object of type 'WebElement' has no len()
CSV 저장 완료: melon_신의 키스_songs.csv
총 0곡 수집 완료


""


In [15]:

def extract_current_page(driver, wait):
    data = []

    try:
        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )
    except TimeoutException:
        return []

    rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")

        if len(cols) < 5:
            continue

        title_lines = [
            line.strip()
            for line in cols[2].text.split("\n")
            if line.strip()
        ]

        title = ""

        for line in title_lines:
            if line.startswith("Title "):
                title = line.replace("Title ", "").strip()
                break

        if not title:
            for line in title_lines:
                if (
                    "재생" not in line
                    and "담기" not in line
                    and "상세정보" not in line
                    and not line.startswith("Title")
                ):
                    title = line
                    break

        artist_lines = [
            line.strip()
            for line in cols[3].text.split("\n")
            if line.strip()
        ]
        artist = artist_lines[0] if artist_lines else ""

        album_lines = [
            line.strip()
            for line in cols[4].text.split("\n")
            if line.strip()
        ]
        album = album_lines[0] if album_lines else ""

        try:
            like = row.find_element(
                By.CSS_SELECTOR,
                "button.like span.cnt"
            ).text.strip()
        except:
            like = ""

        if title:
            data.append({
                "곡명": title,
                "아티스트": artist,
                "앨범": album,
                "좋아요수": like
            })

    return data

In [16]:
def melon_search_all_pages(keyword, max_page=30):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    all_data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )

        song_tab.click()
        time.sleep(3)

        for page in range(1, max_page + 1):
            page_data = extract_current_page(driver, wait)

            if not page_data:
                print(f"{page}페이지 데이터가 없어 종료합니다.")
                break

            all_data.extend(page_data)
            print(f"{page}페이지 크롤링 완료: {len(page_data)}곡")

            next_start_index = page * 50 + 1

            try:
                driver.execute_script(
                    f"pageObj.sendPage('{next_start_index}');"
                )
                time.sleep(3)

            except Exception as e:
                print("다음 페이지 이동 실패:", e)
                break

        df = pd.DataFrame(all_data)

        if not df.empty:
            df = df.drop_duplicates(
                subset=["곡명", "아티스트", "앨범"]
            )
            df.index = df.index + 1

        file_name = f"melon_{keyword}_all_songs.csv"

        df.to_csv(
            file_name,
            encoding="utf-8-sig"
        )

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

In [17]:
melon_search_all_pages("조용필", 30)

1페이지 크롤링 완료: 50곡
2페이지 크롤링 완료: 50곡
3페이지 크롤링 완료: 50곡
4페이지 크롤링 완료: 50곡
5페이지 크롤링 완료: 50곡
6페이지 크롤링 완료: 50곡
7페이지 크롤링 완료: 50곡
8페이지 크롤링 완료: 50곡
9페이지 크롤링 완료: 50곡
10페이지 크롤링 완료: 50곡
11페이지 크롤링 완료: 50곡
12페이지 크롤링 완료: 50곡
13페이지 크롤링 완료: 50곡
14페이지 크롤링 완료: 50곡
15페이지 크롤링 완료: 50곡
16페이지 크롤링 완료: 50곡
17페이지 크롤링 완료: 50곡
18페이지 크롤링 완료: 50곡
19페이지 크롤링 완료: 50곡
20페이지 크롤링 완료: 50곡
21페이지 크롤링 완료: 50곡
22페이지 크롤링 완료: 50곡
23페이지 크롤링 완료: 50곡
24페이지 크롤링 완료: 50곡
25페이지 크롤링 완료: 50곡
26페이지 크롤링 완료: 20곡
27페이지 데이터가 없어 종료합니다.
CSV 저장 완료: melon_조용필_all_songs.csv
총 1270곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,바람의 노래,조용필,조용필 16집,"26,576"
2,꿈,조용필,The Dreams,"16,275"
3,HOT 잊혀진 사랑,조용필,30주년 기념 음반 Part 1,"2,414"
4,Bounce,조용필,Hello,"40,021"
5,HOT 단발머리,조용필,조용필 1집 (Remastered),"11,088"
...,...,...,...,...
1266,님이여,조용필,기쁜 우리 젊은날의 가요 4집,15
1267,마음속의 그림자,조용필,기쁜 우리 젊은날의 가요 4집,14
1268,세월이 가면,조용필,기쁜 우리 젊은날의 가요 1집,24
1269,Bounce - 조용필 (MR),음악상자,음악상자 12집,3


# 3. 스타벅스 서울 전체 매장 크롤링

In [4]:
import re
from selenium.webdriver import ActionChains
from bs4 import BeautifulSoup


In [ ]:
def fetch_starbucks():
    url = "https://www.starbucks.co.kr/index.do"
    driver = webdriver.Chrome()
    driver.maximize_window()
    driver.get(url)
    time.sleep(2)
    
    # 메뉴 이동
    action = ActionChains(driver)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03 > div > div > div > ul:nth-child(1) > li:nth-child(3) > a"
    )

    action.move_to_element(first_tag) \
          .move_to_element(second_tag) \
          .click() \
          .perform()

    # 서울 선택
    seoul_tag = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a"
        ))
    )

    seoul_tag.click()

    # 구 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "set_gugun_cd_btn")
        )
    )

    gu_elements = driver.find_elements(
        By.CLASS_NAME,
        "set_gugun_cd_btn"
    )

    # 전체 선택
    gu_elements[0].click()

    # 매장 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "quickResultLstCon")
        )
    )

    # HTML 가져오기
    req = driver.page_source

    soup = BeautifulSoup(req, "html.parser")

    stores = soup.find(
        'ul',
        'quickSearchResultBoxSidoGugun'
    ).find_all('li')
 
    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    lat_list = []
    lng_list = []

    # 데이터 추출
    for store in stores:

        store_name = store.find("strong").text

        store_addr = store.find("p").text

        # 전화번호 제거
        store_addr = re.sub(
            # r(raw string) : 백슬래시 \를 특별한 이스케이프 문자로 처리하지 않고 그대로 문자열에 넣음 
            r'\d{4}-\d{4}$',
            '',
            store_addr
        ).strip()

        store_lat = store['data-lat']
        store_lng = store['data-long']

        store_list.append(store_name)
        addr_list.append(store_addr)
        lat_list.append(store_lat)
        lng_list.append(store_lng)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'store': store_list,
        'addr': addr_list,
        'lat': lat_list,
        'lng': lng_list
    })

    driver.quit()

    return df


# 함수 실행
starbucks_df = fetch_starbucks()

# CSV 저장
starbucks_df.to_csv(
    "starbucks_seoul.csv",
    index=False,
    encoding='utf-8-sig'
)

print("데이터가 starbucks_seoul.csv 파일로 저장되었습니다.")
print(starbucks_df.head())

데이터가 starbucks_seoul.csv 파일로 저장되었습니다.
       store                        addr         lat          lng
0  역삼아레나빌딩       서울특별시 강남구 언주로 425 (역삼동)   37.501087   127.043069
1   논현역사거리      서울특별시 강남구 강남대로 538 (논현동)   37.510178   127.022223
2  신사역성일빌딩      서울특별시 강남구 강남대로 584 (논현동)  37.5139309  127.0206057
3   국기원사거리      서울특별시 강남구 테헤란로 125 (역삼동)   37.499517   127.031495
4   대치재경빌딩    서울특별시 강남구 남부순환로 2947 (대치동)   37.494668   127.062583


In [68]:
import time
import pandas as pd
import os
from urllib.parse import urljoin, urlparse
import re
from bs4 import BeautifulSoup
import requests
from selenium.webdriver import ActionChains
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [ ]:
from asyncio import wait


def fetch_banapresso():
    url = "https://www.banapresso.com/"
    
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    driver.get(url)
    time.sleep(2)

    action = ActionChains(driver)
    wait = WebDriverWait(driver, 10)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2)"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a"
    )

    action.move_to_element(first_tag)\
          .move_to_element(second_tag)\
          .click()\
          .perform()
          
    # 매장 목록이 화면에 나타날 때까지 대기
    wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, ".store_name_map")
        )
    )      
        
    before_count = 0

    while True:
        store_names = driver.find_elements(
            By.CSS_SELECTOR,
            ".store_name_map .name"
        )

        current_count = len(store_names)
        print(f"현재 로딩된 매장 수: {current_count}")

        if current_count == before_count:
            break

        before_count = current_count

        driver.execute_script(
            """
            const listBox = document.querySelector('.store_shop_list');
            if (listBox) {
                listBox.scrollTop = listBox.scrollHeight;
            }
            """
        )

        time.sleep(1)
    
    # target = driver.find_element(By.CSS_SELECTOR, ".observer")

    # driver.execute_script(
    # "arguments[0].scrollIntoView();",
    # target
    # )
    # time.sleep(1)
    
    req = driver.page_source
    soup = BeautifulSoup(req, "html.parser")
    
    stores = soup.select(".store_name_map")
    store_data = []

    for store in stores:
        name_tag = store.select_one(".name")
        address_tag = store.select_one(".address")
        time_tag = store.select_one(".store-time-wrap")
        store_parking_tag = store.select_one(".parking")

        store_name = name_tag.get_text(strip=True) if name_tag else ""
        store_address = address_tag.get_text(strip=True) if address_tag else ""
        store_time = time_tag.get_text(" ", strip=True) if time_tag else ""
        store_parking = store_parking_tag.get_text(strip=True) if store_parking_tag else ""

        if store_name and store_address:
            store_data.append({
                "매장명": store_name,
                "주소": store_address,
                "영업정보": store_time,
                "주차 정보": store_parking
            })

    df = pd.DataFrame(store_data)
    df = df.drop_duplicates(subset=["매장명", "주소"])
    df.index = df.index + 1


    driver.quit()
    
    return df


banapresso_df = fetch_banapresso()

banapresso_df.to_csv(
    "banapresso.csv",
    index=False,
    encoding='utf-8-sig'
)
    


현재 로딩된 매장 수: 10
현재 로딩된 매장 수: 20
현재 로딩된 매장 수: 30
현재 로딩된 매장 수: 40
현재 로딩된 매장 수: 50
현재 로딩된 매장 수: 60
현재 로딩된 매장 수: 70
현재 로딩된 매장 수: 80
현재 로딩된 매장 수: 90
현재 로딩된 매장 수: 100
현재 로딩된 매장 수: 110
현재 로딩된 매장 수: 120
현재 로딩된 매장 수: 130
현재 로딩된 매장 수: 140
현재 로딩된 매장 수: 150
현재 로딩된 매장 수: 160
현재 로딩된 매장 수: 170
현재 로딩된 매장 수: 180
현재 로딩된 매장 수: 190
현재 로딩된 매장 수: 200
현재 로딩된 매장 수: 210
현재 로딩된 매장 수: 220
현재 로딩된 매장 수: 228
현재 로딩된 매장 수: 228
            매장명                              주소  \
1          가락몰점   서울특별시 송파구 양재대로 932, 업무동 1층 로비   
2     가산디지털단지역점                서울시 금천구 가산동 60-3   
3        가산안양천점   서울 금천구 가산 디지털2로 127-143, 101호   
4       가산어반워크점    서울시 금천구 가산디지털2로 135, 1동 142호   
5     가산우림라이온스점        서울 금천구 가산디지털1로 168 b131호   
..          ...                             ...   
223   홍대입구역사거리점                  서울 마포구 양화로 129   
224     회기역사거리점         서울 동대문구 회기로 176 (회기동81)   
225       AK금정점  경기도 군포시 금정동 689번지 AK플라자 금정점 2층   
226  가산에이스비즈포레점               서울 금천구 가산동 459-23   
227  가산한라시그마밸리점    서울특별시 금천구 가산디지털2로 53 

In [66]:
banapresso_df

,매장명,주소,영업정보,주차 정보
1,가락몰점,"서울특별시 송파구 양재대로 932, 업무동 1층 로비",OPEN 07:30~23:30,
2,가산디지털단지역점,서울시 금천구 가산동 60-3,CLOSE 07:00~19:00,
3,가산안양천점,"서울 금천구 가산 디지털2로 127-143, 101호",OPEN 07:00~20:00,
4,가산어반워크점,"서울시 금천구 가산디지털2로 135, 1동 142호",CLOSE 07:00~17:30,
5,가산우림라이온스점,서울 금천구 가산디지털1로 168 b131호,CLOSE 07:00~17:00,
...,...,...,...,...
223,홍대입구역사거리점,서울 마포구 양화로 129,OPEN 07:00~21:00 주차불가,주차불가
224,회기역사거리점,서울 동대문구 회기로 176 (회기동81),OPEN 07:00~22:00,
225,AK금정점,경기도 군포시 금정동 689번지 AK플라자 금정점 2층,OPEN 07:30~22:30 지하 주차장 이용 가능,지하 주차장 이용 가능
226,가산에이스비즈포레점,서울 금천구 가산동 459-23,CLOSE,


In [20]:
def yes24_crawling(keyword, page_count):
    data = []

    os.makedirs("images/yes24", exist_ok=True)

    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    try:
        driver.get("https://www.yes24.com/Main/default.aspx")
        time.sleep(2)

        try:
            search_box = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "#query"))
            )
            search_box.clear()
            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)
            time.sleep(3)
        except TimeoutException:
            driver.get(
                f"https://www.yes24.com/Product/Search?domain=ALL&query={keyword}"
            )
            time.sleep(3)

        for page in range(1, page_count + 1):
            print(f"[YES24] {page}페이지 크롤링 중")

            try:
                wait.until(
                    EC.presence_of_all_elements_located(
                        (By.CSS_SELECTOR, "li[data-goods-no], div.itemUnit, li.goodsData")
                    )
                )
            except TimeoutException:
                print("[YES24] 검색 결과가 없거나 페이지 로딩이 지연되었습니다.")
                break

            books = driver.find_elements(
                By.CSS_SELECTOR,
                "li[data-goods-no], div.itemUnit, li.goodsData"
            )

            if len(books) == 0:
                print("[YES24] 더 이상 수집할 책이 없습니다.")
                break

            before_count = len(data)

            for index, book in enumerate(books, start=1):
                try:
                    try:
                        title = book.find_element(By.CSS_SELECTOR, ".gd_name").text.strip()
                    except:
                        title = book.find_element(
                            By.CSS_SELECTOR,
                            "a[href*='/Product/Goods/']"
                        ).text.strip()

                    if title == "":
                        continue

                    try:
                        author = book.find_element(By.CSS_SELECTOR, ".info_auth").text.strip()
                    except:
                        author = ""

                    try:
                        price = book.find_element(By.CSS_SELECTOR, ".yes_b").text.strip()
                    except:
                        price = ""

                    try:
                        publisher = book.find_element(By.CSS_SELECTOR, ".info_pub").text.strip()
                    except:
                        publisher = ""

                    try:
                        pub_date = book.find_element(By.CSS_SELECTOR, ".info_date").text.strip()
                    except:
                        pub_date = ""

                    image_path = ""

                    try:
                        img_tag = book.find_element(By.CSS_SELECTOR, "img")
                        img_url = img_tag.get_attribute("src")

                        if img_url:
                            img_response = requests.get(
                                img_url,
                                headers={"User-Agent": "Mozilla/5.0"},
                                timeout=10
                            )

                            ext = os.path.splitext(urlparse(img_url).path)[1]
                            if ext.lower() not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
                                content_type = img_response.headers.get("Content-Type", "")
                                if "png" in content_type:
                                    ext = ".png"
                                elif "gif" in content_type:
                                    ext = ".gif"
                                elif "webp" in content_type:
                                    ext = ".webp"
                                else:
                                    ext = ".jpg"

                            safe_title = re.sub(r'[\\/:*?"<>|]', "_", title)[:60]
                            file_name = f"yes24_{len(data) + 1}_{safe_title}{ext}"
                            image_path = os.path.join("images", "yes24", file_name)

                            with open(image_path, "wb") as file:
                                file.write(img_response.content)
                    except Exception as error:
                        print("[YES24] 이미지 저장 실패:", error)

                    data.append({
                        "검색어": keyword,
                        "책제목": title,
                        "저자": author,
                        "가격": price,
                        "출판사": publisher,
                        "출판일": pub_date,
                        "이미지": image_path
                    })

                except Exception as error:
                    print("[YES24] 요소 추출 실패:", error)

            if len(data) == before_count:
                print("[YES24] 현재 페이지에서 수집된 데이터가 없어 종료합니다.")
                break

            if page == page_count:
                break

            try:
                next_page = page + 1
                page_link = driver.find_element(By.LINK_TEXT, str(next_page))
                driver.execute_script("arguments[0].click();", page_link)
                time.sleep(3)
            except:
                driver.get(
                    f"https://www.yes24.com/Product/Search?domain=ALL&query={keyword}&page={page + 1}"
                )
                time.sleep(3)

    except Exception as error:
        print("[YES24] 크롤링 중 오류 발생:", error)

    finally:
        driver.quit()

    return data

In [23]:
yes24_crawling("asdasdasdasdsaasdasd", 1)

[YES24] 1페이지 크롤링 중
[YES24] 검색 결과가 없거나 페이지 로딩이 지연되었습니다.


[]

In [ ]:
def kyobo_crawling(keyword, page_count):
    data = []

    os.makedirs("images/kyobo", exist_ok=True)

    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    try:
        driver.get("https://www.kyobobook.co.kr/")
        time.sleep(2)
        try:
            search_box = wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "#searchKeyword"))
            )
            search_box.clear()
            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)
            time.sleep(3)

            if keyword not in driver.page_source:
                driver.get(
                    f"https://search.kyobobook.co.kr/search?keyword={keyword}&gbCode=TOT&target=total"
                )
                time.sleep(3)
        except TimeoutException:
            # 검색창을 찾지 못한 경우 검색 결과 URL로 직접 이동한다.
            driver.get(
                f"https://search.kyobobook.co.kr/search?keyword={keyword}&gbCode=TOT&target=total"
            )
            time.sleep(3)

        for page in range(1, page_count+1):
            print(f"{page}페이지 크롤링 중")

            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, ".prod_item")))

            except TimeoutException:
                print("검색 결과가 없거나 페이지 로딩이 지연되었습니다.")
                break

            books = driver.find_elements(By.CSS_SELECTOR, ".prod_item")

            if len(books) == 0:
                print("더 이상 수집할 책이 없습니다.")
                break

            before_count = len(data)

            for book in books:
                try:
                    try:
                        title = book.find_element(
                            By.CSS_SELECTOR, "a.prod_info").text.strip()
                    except:
                        title = book.find_element(
                            By.CSS_SELECTOR, "a[href*='product']").text.strip()

                    if title == "":
                        continue

                    try:
                        author = book.find_element(
                            By.CSS_SELECTOR, "a.author").text.strip()
                    except:
                        author = ""

                    try:
                        price = book.find_element(
                            By.CSS_SELECTOR, ".price").text.strip()
                    except:
                        price = ""

                    try:
                        publish = book.find_element(
                            By.CSS_SELECTOR, ".prod_publish").text.strip()
                    except:
                        publish = ""
                    
                    publish_lines = publish.splitlines()
                    publisher = publish_lines[0] if len(publish_lines) > 0 else ""
                    publish_data = publish_lines[1] if len(publish_lines) > 1 else ""
                    
                    image_path = ""
                    
                    try:
                        image_tag = book.find_element(By.CSS_SELECTOR, "img")
                        image_url = image_tag.get_attribute("src")
                        
                        if image_url:
                            image_respons = requests.get(
                                image_url,
                                headers={"User-Agent": "Mozilla/5.0"},
                                timeout=10
                            )
                        extension = os.path.splitext(urlparse(image_url).path)[1] 
                        if extension.lower() not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
                            content_type = image_respons.headers.get("Content-Type", "")
                            if "png" in content_type:
                                extension = ".png"
                            elif "gif" in content_type:
                                extension = ".gif"
                            elif "webp" in content_type:
                                extension = ".webp"
                            else:
                                extension = ".jpg"
                        
                        file_name = f"{len(data) + 1}_{title}{extension}"
                        image_path = os.path.join("images", "kyobo", file_name)
                        
                        with open(image_path, "wb") as file:
                            file.write(image_respons.content)
                    
                    except Exception as error:
                        print("이미지 크롤링 실패", error)
                    
                    data.append({
                        "검색어": keyword,
                        "책제목": title,
                        "저자": author,
                        "가격": price,
                        "출판사": publisher,
                        "출판일": publish_data,
                        "이미지": image_path
                    })

                except Exception as error:
                    print("크롤링 실패", error)
            
            if len(data) == before_count:
                print("현재 페이지에서 데이터 없음")
                break
        
            if page == page_count:
                break
            
            try:
                next_page = page + 1
                page_link = driver.find_element(By.LINK_TEXT, str(next_page))
                driver.execute_script("arguments[0].click():", page_link)
                time.sleep(3)
            except:
                driver.get(
                    f"https://search.kyobobook.co.kr/search?keyword={keyword}&gbCode=TOT&target=total&page={page + 1}"
                )
                time.sleep(3)
                
                
    except Exception as error:
        print("크롤링 실패", error)
    
    finally:
        driver.quit()

    return data

In [89]:
data = kyobo_crawling("파이썬", 5)
df = pd.DataFrame(data)
df

1페이지 크롤링 중
2페이지 크롤링 중
3페이지 크롤링 중
4페이지 크롤링 중
5페이지 크롤링 중


,검색어,책제목,저자,가격,출판사,출판일,이미지
0,파이썬,[국내도서] 초보자를 위한 파이썬(Python) 200제,장삼용,"18,000 원",정보문화사,2017년 02월 27일,images/1_[국내도서] 초보자를 위한 파이썬(Python) 200제.jpg
1,파이썬,[eBook] 파이썬 Python 기초 가이드 - 이 책 한 권이면 끝!,박빈,"8,010 원",와이웨이브이퍼블리싱,2025년 02월 20일,images/2_[eBook] 파이썬 Python 기초 가이드 - 이 책 한 권이면...
2,파이썬,[eBook] 누구나 할 수 있다 파이썬 Python 기초,박빈,"8,010 원",와이웨이브이퍼블리싱,2024년 03월 22일,images/3_[eBook] 누구나 할 수 있다 파이썬 Python 기초.jpg
3,파이썬,[국내도서] 작심 3일 파이썬 Python,황덕창,"16,200 원",스포트라잇북,2019년 04월 25일,images/4_[국내도서] 작심 3일 파이썬 Python.jpg
4,파이썬,[국내도서] Black Hat Python,저스틴 지이츠,"25,200 원",에이콘출판,2022년 04월 30일,images/5_[국내도서] Black Hat Python.jpg
...,...,...,...,...,...,...,...
95,파이썬,[국내도서] 코딩 자율학습 잔재미코딩의 파이썬 데이터 분석 입문,Dave Lee,"23,400 원",길벗,2025년 04월 01일,images/96_[국내도서] 코딩 자율학습 잔재미코딩의 파이썬 데이터 분석 입문.jpg
96,파이썬,[국내도서] 데이터 과학 기반의 파이썬 빅데이터 분석,이지영,"31,000 원",한빛아카데미,2024년 11월 25일,images/97_[국내도서] 데이터 과학 기반의 파이썬 빅데이터 분석.jpg
97,파이썬,[국내도서] LUVIT EPL과 유튜브 데이터로 배우는 DuckDB,이기준,"27,000 원",제이펍,2026년 06월 25일,images/98_[국내도서] LUVIT EPL과 유튜브 데이터로 배우는 DuckD...
98,파이썬,[국내도서] 파워 유저를 위한 파이썬 EXPRESS,천인국,"35,000 원",생능출판,2026년 01월 09일,images/99_[국내도서] 파워 유저를 위한 파이썬 EXPRESS.jpg


In [ ]:
def aladin_crawling(keyword, page_count):
    data = []
    
    os.makedirs("images/aladin", exist_ok=True)
    
    options = Options()
    options.add_argument("--start-maximized")
    
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 10)
    
    try:
        driver.get("https://www.aladin.co.kr/home/welcome.aspx")
        time.sleep(2)
        
        try:
            search_box = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "#SearchWord"))
            )
            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)
            time.sleep(3)
            
        except TimeoutException:
            driver.get(
                f"https://www.aladin.co.kr/search/wsearchresult.aspx?SearchTarget=All&SearchWord={keyword}"
            )
            time.sleep(3)
        
        for page in range(1, page_count+1):
            print(f"{page}페이지 크롤링 중")
            
            try:
                wait.until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ss_book_box"))
                )
            except TimeoutException:
                print("검색 결과가 없거나 페이지 로딩이 지연되었습니다.")
                break
                
            books = driver.find_elements(By.CSS_SELECTOR, ".ss_book_box")
            
            if len(books) == 0:
                print("크롤링 할 책이 없습니다.")
                break
            
            before_count = len(data)
            
            for book in books:
                try:
                    title = ""

                    title_tag = book.find_elements(By.CSS_SELECTOR, ".bo3")
                    if len(title_tag) > 0:
                        title = title_tag[0].text.strip()
                        
                    if title == "":
                        title_tag = book.find_elements(By.CSS_SELECTOR, "a[herf*='ItemId']")
                        if len(title_tag) > 0:
                            title = title_tag[0].text.strip()
                    
                    if title == "":
                        continue       
                                       
                    try:
                        info_items = book.find_elements(By.CSS_SELECTOR, ".ss_book_list li")

                        for item in info_items:
                            info_line = item.text.strip()
                            info_parts = [
                                part.strip()
                                for part in info_line.split("|")
                                if part.strip()
                            ]

                            if len(info_parts) >= 3:
                                author = info_parts[0]
                                publisher = info_parts[1]
                                publish_date = info_parts[2]
                                break
                    except:
                        author = ""
                        publisher = ""
                        publish_date = ""
                                                   
                    try:
                        price = book.find_element(By.CSS_SELECTOR, ".ss_p2").text.strip()
                    except:
                        price = ""
                    
                    image_path = ""
                    
                    try:
                        image_tag = book.find_element(By.CSS_SELECTOR, "img")
                        image_url = image_tag.get_attribute("src")
                        
                        if image_url:
                            image_url = urljoin("https://www.aladin.co.kr", image_url)
                            
                            image_response = requests.get(
                                image_url,
                                headers={"User-Agent": "Mozilla/5.0"},
                                timeout=10
                            )
                            extension = os.path.splitext(urlparse(image_url).path)[1]
                            if extension.lower() not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
                                content_type = image_response.headers.get("Content-Type", "")
                                if "png" in content_type:
                                    extension = ".png"
                                elif "gif" in content_type:
                                    extension = ".gif"
                                elif "webp" in content_type:
                                    extension = ".webp"
                                else:
                                    extension = ".jpg"
                            
                            file_name = f"{len(data)+1}_{title}{extension}"
                            image_path = os.path.join("images", "aladin", file_name)
                            
                            with open(image_path, "wb") as file:
                                file.write(image_response.content)
                    except Exception as error:
                        print("이미지 저장 실패", error)
                        break
                    
                    data.append({
                        "검색어": keyword,
                        "책제목": title,
                        "저자": author,
                        "가격": price,
                        "출판사": publisher,
                        "출판일": publish_date,
                        "이미지": image_path
                    })
                except Exception as error:
                    print("크롤링 실패", error)
            
            if len(data) == before_count:
                print("현재 페에지에는 데이터가 없습니다")
                break
            
            if page == page_count:
                break
            
            try:
                next_page = page + 1
                page_link = driver.find_element(By.LINK_TEXT, str(next_page))
                driver.execute_script("arguments[0].click();", page_link)
                time.sleep(3)
            except:
                driver.get(
                    f"https://www.aladin.co.kr/search/wsearchresult.aspx?SearchTarget=All&SearchWord={keyword}&page={page + 1}"
                )
                time.sleep(3)

    except Exception as error:
        print("크롤링 중 오류 발생:", error)

    finally:
        driver.quit()

    return data

In [114]:
data = aladin_crawling("파이썬", 1)
df = pd.DataFrame(data)
df

1페이지 크롤링 중


,검색어,책제목,저자,가격,출판사,출판일,이미지
0,파이썬,Do it! 점프 투 파이썬,박응용 (지은이),"19,800원",이지스퍼블리싱,2023년 6월,images/aladin/1_Do it! 점프 투 파이썬.jpg
1,파이썬,혼자 공부하는 파이썬,윤인성 (지은이),"19,800원",한빛미디어,2022년 6월,images/aladin/2_혼자 공부하는 파이썬.jpg
2,파이썬,파이썬 텍스트 코딩 워크북,"정재웅, 김성안, 박수빈, 배효정, 서정민, 서진원, 양채윤, 이민혁, 장성혜, 최...","4,500원",삼양미디어,2022년 5월,images/aladin/3_파이썬 텍스트 코딩 워크북.jpg
3,파이썬,"파이썬 기초 문법 : 파이썬을 이용한 빅데이터 수집, 분석과 시각화 구독자용",이원하 (지은이),0원,비팬북스,2017년 7월,"images/aladin/4_파이썬 기초 문법 : 파이썬을 이용한 빅데이터 수집, ..."
4,파이썬,이것이 취업을 위한 코딩 테스트다 with 파이썬,나동빈 (지은이),"30,600원",한빛미디어,2020년 8월,images/aladin/5_이것이 취업을 위한 코딩 테스트다 with 파이썬.jpg
5,파이썬,코딩 자율학습 나도코딩의 파이썬 입문,나도코딩 (지은이),"21,600원",길벗,2023년 2월,images/aladin/6_코딩 자율학습 나도코딩의 파이썬 입문.jpg
6,파이썬,한번보고 만드는 AI가 코딩해주는 파이썬,"류태선, 오근철, 안가영, 조예찬 (지은이)","19,800원",IMK,2026년 4월,images/aladin/7_한번보고 만드는 AI가 코딩해주는 파이썬.jpg
7,파이썬,2026 시나공 빅데이터분석기사 실기 Python,김태헌 (지은이),"25,600원",길벗,2026년 4월,images/aladin/8_2026 시나공 빅데이터분석기사 실기 Python.jpg
8,파이썬,모두의 데이터 분석 with 파이썬,"송석리, 이현아 (지은이)","16,200원",길벗,2019년 4월,images/aladin/9_모두의 데이터 분석 with 파이썬.jpg
9,파이썬,난생처음 파이썬 프로그래밍,"우재남, 최민아 (지은이)","24,000원",한빛아카데미(교재),2021년 6월,images/aladin/10_난생처음 파이썬 프로그래밍.jpg
